# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 7: Risk, Likelihood, and Estimation

This notebook connects Chapters 3--5. A bounded loss turns population risk
into an estimand and empirical risk into an estimator. We then estimate an
entire distribution function and construct a simultaneous DKW band.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(707)


## Using sample risk to estimate population risk

Specify an i.i.d. binary-classification model as follows:

- $X\sim\mathrm{Uniform}([-1,1])$;
- given $X=x$, $Y\sim\mathrm{Bernoulli}(\eta(x))$, where
  $\eta(x)=0.2$ for $x\leq0$ and $0.8$ for $x>0$;
- use the fixed classifier $g(x)=\mathbf 1_{\{x>0\}}$ and 0--1 loss.

The loss $L=\mathbf 1_{\{Y\ne g(X)\}}$ is Bernoulli with mean
$R(g)=0.2$. Therefore, for an i.i.d. sample, empirical risk is an unbiased
estimator of population risk, with standard error
$\sqrt{0.2(0.8)/n}$. Hoeffding gives a distribution-free finite-sample
confidence interval because each loss lies in $[0,1]$.


In [ ]:
def simulate_empirical_risks(n, repetitions, generator):
    x = generator.uniform(-1, 1, size=(repetitions, n))
    eta = np.where(x > 0, 0.8, 0.2)
    y = generator.random(size=(repetitions, n)) < eta
    prediction = x > 0
    return np.mean(y != prediction, axis=1)


risk = 0.20
alpha = 0.05
for n in (25, 100, 400):
    empirical_risks = simulate_empirical_risks(n, 5000, rng)
    delta = np.sqrt(np.log(2 / alpha) / (2 * n))
    lower = np.maximum(0, empirical_risks - delta)
    upper = np.minimum(1, empirical_risks + delta)
    print(
        f"n={n:3d}: mean={empirical_risks.mean():.4f}, "
        f"SE={empirical_risks.std(ddof=0):.4f}, "
        f"theory SE={np.sqrt(risk*(1-risk)/n):.4f}, "
        f"coverage={np.mean((lower <= risk) & (risk <= upper)):.4f}"
    )


This calculation is for one **fixed** classifier. If a large class is searched
using the same sample, a pointwise Hoeffding bound for each fixed rule does
not automatically control the selected rule. Uniform control over a class is
a separate question.


## Estimating a distribution function from data

For i.i.d. observations with unknown CDF $F$, the empirical CDF is

$$
\widehat F_n(t)=\frac1n\sum_{i=1}^n\mathbf 1_{\{X_i\leq t\}}.
$$

For each fixed $t$, it is an unbiased estimator of $F(t)$, with variance
$F(t)(1-F(t))/n$. The Dvoretzky--Kiefer--Wolfowitz inequality makes the
stronger simultaneous statement

$$
\mathbb P\left(\sup_t|\widehat F_n(t)-F(t)|>\varepsilon\right)
\leq 2e^{-2n\varepsilon^2}.
$$

No continuity, density, or moment assumption on $F$ is required.


In [ ]:
n = 150
sample = rng.exponential(scale=1.0, size=n)
grid = np.linspace(-0.25, 5, 800)
ecdf = np.mean(sample[:, None] <= grid[None, :], axis=0)
true_cdf = np.where(grid < 0, 0.0, 1 - np.exp(-grid))

epsilon_dkw = np.sqrt(np.log(2 / alpha) / (2 * n))
lower_band = np.maximum(0.0, ecdf - epsilon_dkw)
upper_band = np.minimum(1.0, ecdf + epsilon_dkw)

fig, ax = plt.subplots(figsize=(7, 4))
ax.step(grid, ecdf, where="post", label="ECDF")
ax.plot(grid, true_cdf, color="black", label="true exponential CDF")
ax.fill_between(grid, lower_band, upper_band, step="post", alpha=0.25,
                label="95% DKW band")
ax.set(xlabel="t", ylabel="CDF", ylim=(-0.03, 1.03))
ax.legend()
plt.show()


In [ ]:
# For a continuous CDF, the exact sup distance is attained next to a sample jump.
ordered = np.sort(sample)
f_at_ordered = 1 - np.exp(-ordered)
indices = np.arange(1, n + 1)
d_plus = np.max(indices / n - f_at_ordered)
d_minus = np.max(f_at_ordered - (indices - 1) / n)
d_sup = max(d_plus, d_minus)

print(f"exact sup distance for this sample = {d_sup:.4f}")
print(f"DKW half-width = {epsilon_dkw:.4f}")
print("true CDF is inside this band everywhere?", d_sup <= epsilon_dkw)


Because the DKW upper bound tends to zero for every fixed
$\varepsilon>0$, it proves uniform consistency in probability:
$\|\widehat F_n-F\|_\infty\to0$. This is different from checking one
chosen value of $t$, and stronger than separately displaying several
pointwise intervals.


## Checkpoint: empirical risk and distribution functions

1. Change the classifier to always predict 1. Derive its population risk
   before estimating it, then compare its empirical risk with the Bayes rule.
2. For a fixed $t$, derive the mean and variance of
   $\widehat F_n(t)$ by identifying its summands as Bernoulli variables.
3. Repeat the DKW plot for a discrete Bernoulli distribution. Do not use the
   continuous-CDF shortcut for the exact supremum; evaluate the step functions
   directly on intervals around their jump points.


## Likelihood and estimation

This notebook accompanies Chapter 4, Section 4.2, and Chapter 5, Sections
5.2--5.3. We treat likelihood as a function of a parameter after data are
observed, connect it to empirical log loss, handle Bernoulli boundary cases,
and connect Gaussian likelihood with linear regression.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import xlogy
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(708)


## Likelihood scores possible parameter values

For an i.i.d. observed sample $z=(z_1,\ldots,z_n)$ from a model with PMF
or density $p_\theta$, the likelihood is

$$
\mathcal L_n(\theta;z)=\prod_{i=1}^n p_\theta(z_i).
$$

The data are held fixed and $\theta$ varies. Unless an additional model
for $\theta$ is introduced, likelihood is not a probability density or
conditional probability for $\theta$. Products can underflow, so numerical
work normally uses the log-likelihood.


## What happens at the edge of the Bernoulli model?

If $z_i\in\{0,1\}$, $k=\sum_i z_i$, and
$0\leq\theta\leq1$, then

$$
\ell_n(\theta)=k\log\theta+(n-k)\log(1-\theta).
$$

We use $0\log0=0$. If a positive count is multiplied by $\log0$, the
log-likelihood is $-\infty$. Consequently the MLE is $k/n$, including
the boundary cases: it is 0 if all observations are 0 and 1 if all are 1.


In [ ]:
def bernoulli_log_likelihood(theta, successes, n):
    theta = np.asarray(theta, dtype=float)
    if np.any((theta < 0) | (theta > 1)):
        raise ValueError("theta must lie in [0, 1]")
    return xlogy(successes, theta) + xlogy(n - successes, 1 - theta)


n = 30
theta_true = 0.30
data = rng.binomial(1, theta_true, size=n)
successes = int(data.sum())
theta_hat = successes / n

theta_grid = np.linspace(0, 1, 1001)
log_likelihood = bernoulli_log_likelihood(theta_grid, successes, n)

fig, ax = plt.subplots(figsize=(7, 3.8))
finite = np.isfinite(log_likelihood)
ax.plot(theta_grid[finite], log_likelihood[finite])
ax.axvline(theta_hat, color="black", linestyle="--", label=f"MLE={theta_hat:.3f}")
ax.set(xlabel=r"candidate $\theta$", ylabel="log-likelihood")
ax.legend()
plt.show()

print("successes =", successes, "out of", n)
print("all-zero sample MLE =", 0 / n)
print("all-one sample MLE =", n / n)


## Why negative log-likelihood gives log loss

For a true $\mathrm{Bernoulli}(p)$ law with $0<p<1$, the population
log-loss risk is

$$
R_p(\theta)=-p\log\theta-(1-p)\log(1-\theta),\qquad0<\theta<1.
$$

It is uniquely minimized at $\theta=p$. Replacing $p$ by the observed
fraction of successes gives empirical log loss; multiplying it by $n$
gives the negative log-likelihood. Thus maximum likelihood is empirical
risk minimization for log loss in this model.


In [ ]:
p = 0.30
interior_grid = np.linspace(0.001, 0.999, 1000)
population_log_loss = -(xlogy(p, interior_grid) + xlogy(1 - p, 1 - interior_grid))
empirical_log_loss = -bernoulli_log_likelihood(interior_grid, successes, n) / n

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(interior_grid, population_log_loss, label="population log-loss risk")
ax.plot(interior_grid, empirical_log_loss, label="empirical log loss")
ax.axvline(p, color="black", linestyle=":", label="true p")
ax.axvline(theta_hat, color="grey", linestyle="--", label="MLE")
ax.set(xlabel=r"candidate $\theta$", ylabel="log loss", ylim=(0.5, 2.0))
ax.legend()
plt.show()


## How a Gaussian model leads to least squares

Suppose the conditional model is

$$
Y_i\mid X_i=x_i\sim\mathcal N(ax_i+b,\sigma^2)
$$

independently, with a fixed $\sigma>0$. The negative conditional
log-likelihood equals

$$
\frac n2\log(2\pi\sigma^2)
+\frac{1}{2\sigma^2}\sum_{i=1}^n(y_i-(ax_i+b))^2.
$$

The first term is constant in $(a,b)$, so maximum likelihood and least
squares choose the same slope and intercept. The model and independence
assumptions are what justify the likelihood calculation.

Scikit-learn estimators use a common interface: create the estimator, call
`fit(X, y)`, and then call `predict(X_new)`. The feature matrix `X` has
shape `(n_samples, n_features)`. Here `x_reg[:, None]` turns one length-$n$
feature vector into an $n\times1$ matrix. After fitting, `intercept_` and
`coef_` contain the fitted intercept and coefficient.


In [ ]:
n_reg = 100
sigma_data = 0.40
x_reg = rng.uniform(-1, 1, size=n_reg)
y_reg = 1.5 - 2.0 * x_reg + rng.normal(0, sigma_data, size=n_reg)


In [ ]:
# Change only sigma_assumed to compare likelihoods for these fixed data.
sigma_assumed = 0.40
X_reg = x_reg[:, None]

model = LinearRegression()
model.fit(X_reg, y_reg)
prediction = model.predict(X_reg)
sse = np.sum((y_reg - prediction) ** 2)
negative_log_likelihood = (
    n_reg / 2 * np.log(2 * np.pi * sigma_assumed**2)
    + sse / (2 * sigma_assumed**2)
)

print(f"fitted intercept = {model.intercept_:.4f}")
print(f"fitted slope = {model.coef_[0]:.4f}")
print(f"SSE at the fit = {sse:.4f}")
print(f"assumed sigma = {sigma_assumed:.3f}")
print(f"negative conditional log-likelihood = {negative_log_likelihood:.4f}")

plot_grid = np.linspace(-1, 1, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_reg, y_reg, s=18, alpha=0.55, label="observations")
ax.plot(plot_grid, 1.5 - 2 * plot_grid, color="black", label="true regression function")
ax.plot(plot_grid, model.predict(plot_grid[:, None]), color="tab:red", label="fitted line")
ax.set(xlabel="x", ylabel="y")
ax.legend()
plt.show()


## Why the Bernoulli estimate settles near the truth

For i.i.d. Bernoulli observations, the MLE is the sample mean. The strong law
of large numbers therefore gives $\widehat\theta_n\to p$ almost surely.
The plot is one reproducible realization of that theorem, not its proof.


In [ ]:
long_sample = rng.binomial(1, p, size=3000)
running_mle = np.cumsum(long_sample) / np.arange(1, long_sample.size + 1)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(np.arange(1, long_sample.size + 1), running_mle)
ax.axhline(p, color="black", linestyle="--", label="true p")
ax.set(xlabel="n", ylabel=r"$\widehat\theta_n$")
ax.legend()
plt.show()


## Recap

1. For the sample $(0,0,0,0)$, evaluate the Bernoulli likelihood at
   $\theta=0$, $1/2$, and 1. Explain the boundary MLE without
   differentiating.
2. Differentiate $R_p(\theta)$ on $(0,1)$ and verify its unique
   minimizer when $0<p<1$.
3. Change `sigma_assumed` in the Gaussian likelihood cell while holding the
   generated data fixed.
   Explain why the least-squares slope and intercept do not change when
   $\sigma$ is fixed during their optimization.
